# ijson

ijson parses JSON incrementally, as it arrives.

## Example 1 — the import, the version and the backend in use (the yajl2_c C extension)

In [ ]:
import importlib.machinery, importlib.metadata, io
import ijson

print("ijson", ijson.__version__, "-- backend:", ijson.backend_name)
assert ijson.__version__ == importlib.metadata.version("ijson")
# Not the silent pure-Python fallback: the yajl2_c C extension is the backend in use.
assert ijson.backend_name == "yajl2_c", ijson.backend_name
import ijson.backends._yajl2 as c_extension
assert c_extension.__file__.endswith(tuple(importlib.machinery.EXTENSION_SUFFIXES)), c_extension.__file__
assert ijson.ALL_BACKENDS[0] == "yajl2_c" and "python" in ijson.ALL_BACKENDS, ijson.ALL_BACKENDS
assert list(ijson.items(io.BytesIO(b'{"answer": [4, 2]}'), "answer.item")) == [4, 2]
print("OK -- the yajl2_c backend is the default and parses:", c_extension.__file__.rsplit("/", 1)[-1])

## Example 2 — pick values out of a document by prefix, without loading it whole

In [ ]:
import decimal, io
import ijson

DOCUMENT = (b'{"earth": {"europe": [{"name": "Paris", "pop": 2100000, "area": 105.4}, '
            b'{"name": "Rome", "pop": 2800000, "area": 1285.0}], "big": 12345678901234567890}, "ok": true}')
names = list(ijson.items(io.BytesIO(DOCUMENT), "earth.europe.item.name"))
assert names == ["Paris", "Rome"], names
first_city = list(ijson.kvitems(io.BytesIO(DOCUMENT), "earth.europe.item"))[:3]
assert first_city == [("name", "Paris"), ("pop", 2100000), ("area", decimal.Decimal("105.4"))], first_city
assert list(ijson.items(io.BytesIO(DOCUMENT), "earth.big")) == [12345678901234567890]  # beyond 64 bits, kept exact
exact = [type(v) for v in ijson.items(io.BytesIO(b"[1, 2.5, 1e3]"), "item")]
assert exact == [int, decimal.Decimal, decimal.Decimal], exact  # the default keeps decimals exact
floats = list(ijson.items(io.BytesIO(b"[1, 2.5, 1e3]"), "item", use_float=True))
assert floats == [1, 2.5, 1000.0] and type(floats[1]) is float, floats
whole = list(ijson.items(io.BytesIO(DOCUMENT), ""))  # the empty prefix is the whole document
assert whole[0]["ok"] is True and len(whole[0]["earth"]["europe"]) == 2
print("OK -- city names", names, "; a 20-digit integer and decimals stay exact; use_float gives floats")

## Example 3 — the event stream, and the pure-Python backend agrees with the C one

In [ ]:
import io
import ijson

DOCUMENT = b'{"a": [1, {"b": null}], "c": "text", "d": true}'
events = list(ijson.parse(io.BytesIO(DOCUMENT)))
assert events[:4] == [("", "start_map", None), ("", "map_key", "a"), ("a", "start_array", None), ("a.item", "number", 1)]
assert events[-1] == ("", "end_map", None) and ("a.item.b", "null", None) in events, events
basic = list(ijson.basic_parse(io.BytesIO(b'[1, "x", false]')))
assert basic == [("start_array", None), ("number", 1), ("string", "x"), ("boolean", False), ("end_array", None)]
python_backend, c_backend = ijson.get_backend("python"), ijson.get_backend("yajl2_c")
assert python_backend.__name__ == "ijson.backends.python" and c_backend.__name__ == "ijson.backends.yajl2_c"
for document in (DOCUMENT, b'[[], {}, -0.5, 1e400, "\\u00e9\\ud83d\\ude00"]'):
    assert list(python_backend.parse(io.BytesIO(document))) == list(c_backend.parse(io.BytesIO(document))), document
print(f"OK -- {len(events)} events from the C backend; the pure-Python backend yields the same on both documents")

## Example 4 — push parsing: feed chunks as they arrive (the shape vLLM uses)

In [ ]:
import io
import ijson

@ijson.coroutine
def collect(found):
    while True:
        found.append((yield))

found = []
parser = ijson.items_coro(collect(found), "item.id")
for chunk in (b'[{"id": 1}, {"i', b'd": 2}, {"id"', b': 3}]'):  # chunks cut in the middle of keys and values
    parser.send(chunk)
parser.close()
assert found == [1, 2, 3], found
ready = ijson.sendable_list()
stream = ijson.items_coro(ready, "item")
stream.send(b"[10, 2")
assert list(ready) == [10], list(ready)  # 2 is not complete yet: it may be 20, 2.5, ...
del ready[:]
stream.send(b"0, 30]")
stream.close()
assert list(ready) == [20, 30], list(ready)
several = list(ijson.items(io.BytesIO(b'{"a": 1} {"a": 2}'), "", multiple_values=True))
assert several == [{"a": 1}, {"a": 2}], several
print("OK -- ids", found, "from 3 ragged chunks; an unfinished number is held back until its end arrives")

## Example 5 — malformed JSON raises ijson's own errors on both backends

In [ ]:
import io
import ijson

BAD = (b'{"a": [1, 2', b'{"a": tru}', b"[1,]", b'{"a": 1} {"a": 2}')  # the last: two values, not allowed by default
for backend_name in ("yajl2_c", "python"):
    backend = ijson.get_backend(backend_name)
    for document in BAD:
        try:
            list(backend.items(io.BytesIO(document), ""))
            raise AssertionError(f"{backend_name} accepted malformed JSON {document!r}")
        except ijson.JSONError as error:
            assert isinstance(error, (ijson.IncompleteJSONError, ijson.JSONError)), type(error)
try:
    list(ijson.items(io.BytesIO(b'{"a": [1, 2'), "a.item"))
    raise AssertionError("a truncated document ended without an error")
except ijson.IncompleteJSONError as error:
    message = str(error).split("\n")[0]
partial, damage = [], None
try:
    for value in ijson.items(io.BytesIO(b"[1, 2, oops]"), "item"):
        partial.append(value)
except ijson.JSONError as error:
    damage = error
assert partial == [1, 2] and damage is not None, (partial, damage)  # the values before the damage were delivered
print(f"OK -- {len(BAD)} malformed documents refused by both backends ('{message}'); values before the damage arrive first")